In [1]:
import pandas as pd 
import numpy as np 
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [6]:
df.columns

Index(['nazwa', 'marka', 'cena_aktualna', 'dlugosc_rekawa', 'fason_clean',
       'trend_score', 'bawełna_pct', 'poliester_pct', 'elastan_pct',
       'wiskoza_pct', 'wełna_pct', 'len_pct', 'jedwab_pct', 'nylon_pct',
       'poliamid_pct', 'lyocell_pct', 'modal_pct', 'akryl_pct', 'kaszmir_pct',
       'poliuretan_pct', 'natural_material_pct', 'synthetic_material_pct',
       'premium_material_pct'],
      dtype='str')

In [4]:
import pandas as pd 



In [ ]:
#modele do , xgb, lightgb, randomforest

In [5]:
#zmienna do przewidywania cena aktualna 

df = pd.read_csv('../backend/data/dane_do_modelu.csv')

In [81]:
q99 = df['cena_aktualna'].quantile(0.99)

df = df[df['cena_aktualna'] < q99]

In [82]:
target = 'cena_aktualna'
X = df.drop(columns= target)
y = np.log1p(df[target])
#y = df[target]


In [83]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size= 0.2 )

In [84]:
cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

C:\Users\Thinkpad\AppData\Local\Temp\ipykernel_7972\3112228624.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include='object').columns


In [85]:
cat_proces = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [86]:
num_proces = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

In [87]:
preproccesing = ColumnTransformer([
    ('num_processing', num_proces, num_cols),
    ('cat_processing', cat_proces, cat_cols)
])

In [91]:
models = {
    'catboost' : CatBoostRegressor(),
    'XGBRegressor' : XGBRegressor(),
    'Random_forest': RandomForestRegressor(),
    'lightbm': LGBMRegressor()
}

In [92]:

results = []

for name, model in models.items():
    pipeline = Pipeline([
        ('procesowanie', preproccesing),
        ('model', model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)
    y_test_real = np.expm1(y_test)

    mae = mean_absolute_error(y_test_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_real, y_pred))

    #mae = mean_absolute_error(y_test, y_pred)
    #rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results.append((name, mae, rmse))

    results_df = pd.DataFrame(results, columns=['Model', 'MAE', 'RMSE'])
    print(results_df.sort_values('MAE'))

Learning rate set to 0.045305
0:	learn: 0.6438669	total: 2.68ms	remaining: 2.67s
1:	learn: 0.6355343	total: 5.15ms	remaining: 2.57s
2:	learn: 0.6277694	total: 7.6ms	remaining: 2.52s
3:	learn: 0.6212094	total: 10.2ms	remaining: 2.54s
4:	learn: 0.6153862	total: 12.1ms	remaining: 2.4s
5:	learn: 0.6091002	total: 14.5ms	remaining: 2.4s
6:	learn: 0.6038380	total: 17.1ms	remaining: 2.43s
7:	learn: 0.5976152	total: 19.8ms	remaining: 2.46s
8:	learn: 0.5938299	total: 22.4ms	remaining: 2.47s
9:	learn: 0.5894825	total: 24.8ms	remaining: 2.46s
10:	learn: 0.5862633	total: 27.3ms	remaining: 2.46s
11:	learn: 0.5820044	total: 30.2ms	remaining: 2.48s
12:	learn: 0.5782205	total: 32.5ms	remaining: 2.47s
13:	learn: 0.5738432	total: 35.1ms	remaining: 2.47s
14:	learn: 0.5709425	total: 37.6ms	remaining: 2.47s
15:	learn: 0.5670158	total: 40.2ms	remaining: 2.47s
16:	learn: 0.5632895	total: 42.8ms	remaining: 2.48s
17:	learn: 0.5603253	total: 45.3ms	remaining: 2.47s
18:	learn: 0.5574968	total: 47.8ms	remaining: 2

AttributeError: The following error was raised: 'CatBoostRegressor' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order.